<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


<h1>Lab: Improving an RNN Sentiment Classifier with Regularization and Tuning</h1> 


Estimated time needed: **25** minutes


<h2>Learning Objectives</h2>
    
In this lab, you will:
<ul><li> 
Load the trained RNN model checkpoint and establish a baseline accuracy and loss score on the validation set</li><li> Apply dropout regularization to the RNN hidden layers and retrain the model to observe the effect on overfitting behavior</li><li> Compare the effects of dropout and optimizer choice on convergence speed and validation performance with AI-assisted support</li></ul> 


<h2>Table of contents</h2>
<p>In this lab, you will improve an RNN sentiment classifier with dropout regularization, optimizer tuning, and checkpoint-based baseline comparison.</p>
<ol>
<li><a href="#Import-Required-Libraries">Import Required Libraries</a> </li>
<li><a href="#Dropout-Regularization-in-Recurrent-Neural-Networks">Dropout Regularization in Recurrent Neural Networks</a> </li>
<li><a href="#Prepare-Sentiment-Data">Prepare Sentiment Data</a> </li>
<li><a href="#Define-the-RNN-Classifier">Define the RNN Classifier</a> </li>
<li><a href="#Train-and-Load-a-Baseline-Checkpoint">Train and Load a Baseline Checkpoint</a> </li>
<li><a href="#Establish-Baseline-Metrics">Establish Baseline Metrics</a> </li>
<li><a href="#Apply-Dropout-Regularization">Apply Dropout Regularization</a> </li>
<li><a href="#Compare-Optimizer-Choices">Compare Optimizer Choices</a> </li>
<li><a href="#AI-assisted-Convergence-Support">AI-assisted Convergence Support</a> </li>

</ol>

<hr>


## Import Required Libraries


The following required libraries are __not__ pre-installed in the Skills Network Labs environment. __You will need to run the following cells__ to install or import them:


In [1]:
# Install PyTorch

!pip install torch

In [2]:
# Import the libraries we need for this lab

import random
import re
from collections import Counter
from pathlib import Path

import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence
from torch.utils.data import Dataset, DataLoader, random_split

random.seed(2)
torch.manual_seed(2)

/opt/conda/lib/python3.12/site-packages/torch/_subclasses/functional_tensor.py:368: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /__w/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))

<!--Empty Space for separating topics-->


## Dropout Regularization in Recurrent Neural Networks


Dropout regularization randomly disables selected activations during training. This can reduce overfitting by preventing the model from relying too strongly on a small set of hidden features.


<p style="text-align:center">
    <img src="https://commons.wikimedia.org/wiki/Special:Redirect/file/Dropout%20mechanism.png" width="650" alt="Dropout regularization mechanism diagram">
</p>
<p style="text-align:center"><em>Source: Wikimedia Commons, Dropout mechanism.png by Daniel Voigt Godoy</em></p>


For recurrent models, dropout is commonly applied to embeddings, between stacked recurrent layers, or before the final classifier layer.


<!--Empty Space for separating topics-->


## Prepare Sentiment Data


Create and preprocess a small sentiment dataset. The label <code>1</code> represents positive sentiment, and the label <code>0</code> represents negative sentiment:


In [4]:
# Create sentiment examples

positive_sentences = [
    "the product is reliable and easy to use",
    "the support team was helpful and quick",
    "the lesson was clear and practical",
    "the app runs smoothly and looks clean",
    "the package arrived early and intact",
    "the headphones sound rich and balanced",
    "the interface is simple and polished",
    "the service was friendly and efficient",
    "the device feels durable and fast",
    "the course examples were useful and engaging",
]

negative_sentences = [
    "the product is fragile and hard to use",
    "the support team was slow and unhelpful",
    "the lesson was unclear and confusing",
    "the app crashes often and looks messy",
    "the package arrived late and damaged",
    "the headphones sound harsh and distorted",
    "the interface is awkward and frustrating",
    "the service was rude and inefficient",
    "the device feels cheap and slow",
    "the course examples were vague and outdated",
]

base_data = [(sentence, 1) for sentence in positive_sentences] + [(sentence, 0) for sentence in negative_sentences]

Expand the examples with short prefixes to create a larger training set:


In [5]:
# Expand the dataset

prefixes = ["", "overall ", "today ", "in my experience "]
expanded_data = []

for sentence, label in base_data:
    for prefix in prefixes:
        expanded_data.append((prefix + sentence, label))

random.shuffle(expanded_data)

print("Number of examples:", len(expanded_data))
print(expanded_data[:3])

Number of examples: 80
[('the product is reliable and easy to use', 1), ('in my experience the product is reliable and easy to use', 1), ('in my experience the service was rude and inefficient', 0)]

Tokenize the text and build a vocabulary:


In [6]:
# Tokenize text and build vocabulary

def tokenize(text):
    return re.findall(r"\b\w+\b", text.lower())

tokenized_examples = [(tokenize(text), label) for text, label in expanded_data]
token_counts = Counter(token for tokens, label in tokenized_examples for token in tokens)

special_tokens = ["<pad>", "<unk>"]
vocab = {token: index for index, token in enumerate(special_tokens)}

for token in token_counts:
    if token not in vocab:
        vocab[token] = len(vocab)

pad_id = vocab["<pad>"]
unk_id = vocab["<unk>"]

print("Vocabulary size:", len(vocab))

Vocabulary size: 71

Encode the tokenized examples as token ID sequences:


In [7]:
# Encode tokens as token IDs

def encode_tokens(tokens, vocabulary):
    return [vocabulary.get(token, unk_id) for token in tokens]

encoded_examples = [(encode_tokens(tokens, vocab), label) for tokens, label in tokenized_examples]

print(encoded_examples[0])

([2, 3, 4, 5, 6, 7, 8, 9], 1)

Create PyTorch Dataset and DataLoader objects:


In [8]:
# Create Dataset and DataLoader objects

class SentimentDataset(Dataset):
    def __init__(self, examples):
        self.examples = examples

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, index):
        token_ids, label = self.examples[index]
        return torch.tensor(token_ids, dtype=torch.long), torch.tensor(label, dtype=torch.float32)


def collate_batch(batch):
    sequences, labels = zip(*batch)
    lengths = torch.tensor([len(sequence) for sequence in sequences], dtype=torch.long)
    padded_sequences = pad_sequence(sequences, batch_first=True, padding_value=pad_id)
    labels = torch.stack(labels)
    return padded_sequences, labels, lengths


full_dataset = SentimentDataset(encoded_examples)
train_size = int(0.8 * len(full_dataset))
valid_size = len(full_dataset) - train_size

train_dataset, valid_dataset = random_split(
    full_dataset,
    [train_size, valid_size],
    generator=torch.Generator().manual_seed(2),
)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=collate_batch)
valid_loader = DataLoader(valid_dataset, batch_size=8, shuffle=False, collate_fn=collate_batch)

print("Training examples:", len(train_dataset))
print("Validation examples:", len(valid_dataset))

Training examples: 64
Validation examples: 16

<!--Empty Space for separating topics-->


## Define the RNN Classifier


Define a configurable RNN classifier. The model supports stacked recurrent layers and dropout before the final classification layer:


In [9]:
# Define a configurable RNN classifier

class TunableRNNClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, padding_idx, dropout_rate=0.0, num_layers=1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=padding_idx)
        rnn_dropout = dropout_rate if num_layers > 1 else 0.0
        self.rnn = nn.RNN(
            embedding_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=rnn_dropout,
        )
        self.dropout = nn.Dropout(dropout_rate)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, sequences, lengths):
        embedded = self.embedding(sequences)
        packed = pack_padded_sequence(embedded, lengths.cpu(), batch_first=True, enforce_sorted=False)
        packed_output, hidden = self.rnn(packed)
        final_hidden = self.dropout(hidden[-1])
        logits = self.fc(final_hidden)
        return logits.squeeze(1)

Create helper functions for training, evaluation, and experiment tracking:


In [10]:
# Create training and evaluation helpers

def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss = 0
    for sequences, labels, lengths in loader:
        optimizer.zero_grad()
        logits = model(sequences, lengths)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate_model(model, loader, criterion):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for sequences, labels, lengths in loader:
            logits = model(sequences, lengths)
            loss = criterion(logits, labels)
            predictions = (torch.sigmoid(logits) >= 0.5).float()
            total_loss += loss.item()
            correct += (predictions == labels).sum().item()
            total += labels.numel()
    return total_loss / len(loader), correct / total

def run_experiment(dropout_rate=0.0, optimizer_name="Adam", epochs=6):
    torch.manual_seed(2)
    model = TunableRNNClassifier(
        vocab_size=len(vocab),
        embedding_dim=32,
        hidden_dim=32,
        output_dim=1,
        padding_idx=pad_id,
        dropout_rate=dropout_rate,
        num_layers=2,
    )
    criterion = nn.BCEWithLogitsLoss()
    if optimizer_name == "SGD":
        optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
    else:
        optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

    history = []
    for epoch in range(epochs):
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer)
        valid_loss, valid_accuracy = evaluate_model(model, valid_loader, criterion)
        history.append({"epoch": epoch + 1, "train_loss": train_loss, "valid_loss": valid_loss, "valid_accuracy": valid_accuracy})
    return model, history, criterion

<!--Empty Space for separating topics-->


## Train and Load a Baseline Checkpoint


Train a baseline RNN model without dropout, save its checkpoint, and load it back into a new model instance:


In [11]:
# Train and save a baseline checkpoint

baseline_model, baseline_history, criterion = run_experiment(dropout_rate=0.0, optimizer_name="Adam", epochs=6)

checkpoint_path = Path("rnn_sentiment_baseline.pt")
torch.save(
    {
        "model_state_dict": baseline_model.state_dict(),
        "vocab": vocab,
        "history": baseline_history,
    },
    checkpoint_path,
)

print("Checkpoint saved to:", checkpoint_path)

Checkpoint saved to: rnn_sentiment_baseline.pt

Load the checkpoint into a fresh model instance:


In [12]:
# Load the baseline checkpoint

checkpoint = torch.load(checkpoint_path)

loaded_baseline_model = TunableRNNClassifier(
    vocab_size=len(checkpoint["vocab"]),
    embedding_dim=32,
    hidden_dim=32,
    output_dim=1,
    padding_idx=pad_id,
    dropout_rate=0.0,
    num_layers=2,
)
loaded_baseline_model.load_state_dict(checkpoint["model_state_dict"])

print("Checkpoint loaded")

Checkpoint loaded

<!--Empty Space for separating topics-->


## Establish Baseline Metrics


Evaluate the loaded checkpoint on the validation set to establish baseline loss and accuracy:


In [13]:
# Evaluate baseline checkpoint

baseline_valid_loss, baseline_valid_accuracy = evaluate_model(loaded_baseline_model, valid_loader, criterion)

print("Baseline validation loss:", round(baseline_valid_loss, 4))
print("Baseline validation accuracy:", round(baseline_valid_accuracy, 4))

Baseline validation loss: 0.0013
Baseline validation accuracy: 1.0

Inspect baseline convergence history:


In [14]:
# Display baseline history

for row in baseline_history:
    print(row)

{'epoch': 1, 'train_loss': 0.6169616356492043, 'valid_loss': 0.5038268715143204, 'valid_accuracy': 0.8125}
{'epoch': 2, 'train_loss': 0.22079916670918465, 'valid_loss': 0.046822043135762215, 'valid_accuracy': 1.0}
{'epoch': 3, 'train_loss': 0.01912702643312514, 'valid_loss': 0.00819415133446455, 'valid_accuracy': 1.0}
{'epoch': 4, 'train_loss': 0.005219209007918835, 'valid_loss': 0.003001594450324774, 'valid_accuracy': 1.0}
{'epoch': 5, 'train_loss': 0.002322971078683622, 'valid_loss': 0.00178192846942693, 'valid_accuracy': 1.0}
{'epoch': 6, 'train_loss': 0.001508217625087127, 'valid_loss': 0.0013195602223277092, 'valid_accuracy': 1.0}

<!--Empty Space for separating topics-->


## Apply Dropout Regularization


Retrain the RNN with dropout. Compare its validation loss and accuracy with the baseline model:


In [15]:
# Train a dropout-regularized model

dropout_model, dropout_history, dropout_criterion = run_experiment(dropout_rate=0.4, optimizer_name="Adam", epochs=6)
dropout_valid_loss, dropout_valid_accuracy = evaluate_model(dropout_model, valid_loader, dropout_criterion)

print("Dropout validation loss:", round(dropout_valid_loss, 4))
print("Dropout validation accuracy:", round(dropout_valid_accuracy, 4))

Dropout validation loss: 0.0025
Dropout validation accuracy: 1.0

Compare baseline and dropout histories:


In [16]:
# Compare baseline and dropout history

for base_row, dropout_row in zip(baseline_history, dropout_history):
    print(
        "Epoch", base_row["epoch"],
        "baseline valid loss=", round(base_row["valid_loss"], 4),
        "dropout valid loss=", round(dropout_row["valid_loss"], 4),
    )

Epoch 1 baseline valid loss= 0.5038 dropout valid loss= 0.5651
Epoch 2 baseline valid loss= 0.0468 dropout valid loss= 0.3085
Epoch 3 baseline valid loss= 0.0082 dropout valid loss= 0.0521
Epoch 4 baseline valid loss= 0.003 dropout valid loss= 0.0119
Epoch 5 baseline valid loss= 0.0018 dropout valid loss= 0.0044
Epoch 6 baseline valid loss= 0.0013 dropout valid loss= 0.0025

<!--Empty Space for separating topics-->


## Compare Optimizer Choices


Optimizer choice affects convergence speed. Compare Adam and stochastic gradient descent with the same dropout rate:


In [17]:
# Train with SGD for optimizer comparison

sgd_model, sgd_history, sgd_criterion = run_experiment(dropout_rate=0.4, optimizer_name="SGD", epochs=6)
sgd_valid_loss, sgd_valid_accuracy = evaluate_model(sgd_model, valid_loader, sgd_criterion)

print("Adam + dropout validation loss:", round(dropout_valid_loss, 4))
print("Adam + dropout validation accuracy:", round(dropout_valid_accuracy, 4))
print("SGD + dropout validation loss:", round(sgd_valid_loss, 4))
print("SGD + dropout validation accuracy:", round(sgd_valid_accuracy, 4))

Adam + dropout validation loss: 0.0025
Adam + dropout validation accuracy: 1.0
SGD + dropout validation loss: 0.1628
SGD + dropout validation accuracy: 1.0

Create a compact comparison table using Python dictionaries:


In [18]:
# Summarize experiment results

experiment_results = [
    {"model": "Baseline Adam", "dropout": 0.0, "valid_loss": baseline_valid_loss, "valid_accuracy": baseline_valid_accuracy},
    {"model": "Dropout Adam", "dropout": 0.4, "valid_loss": dropout_valid_loss, "valid_accuracy": dropout_valid_accuracy},
    {"model": "Dropout SGD", "dropout": 0.4, "valid_loss": sgd_valid_loss, "valid_accuracy": sgd_valid_accuracy},
]

for result in experiment_results:
    print(result)

{'model': 'Baseline Adam', 'dropout': 0.0, 'valid_loss': 0.0013195602223277092, 'valid_accuracy': 1.0}
{'model': 'Dropout Adam', 'dropout': 0.4, 'valid_loss': 0.002472708933055401, 'valid_accuracy': 1.0}
{'model': 'Dropout SGD', 'dropout': 0.4, 'valid_loss': 0.16277112066745758, 'valid_accuracy': 1.0}

<!--Empty Space for separating topics-->


## AI-assisted Convergence Support


AI-assisted support can help summarize training histories, but the final interpretation should be grounded in validation loss, validation accuracy, and the gap between training and validation behavior:


In [19]:
# Generate a rule-based training interpretation

def interpret_history(history):
    first = history[0]
    last = history[-1]
    train_improved = last["train_loss"] < first["train_loss"]
    valid_improved = last["valid_loss"] <= first["valid_loss"]
    gap = last["valid_loss"] - last["train_loss"]

    if train_improved and valid_improved:
        return "Convergence improved because training and validation loss decreased."
    if train_improved and gap > 0.25:
        return "Possible overfitting: training loss is much lower than validation loss."
    return "The model may need more tuning because validation behavior did not improve clearly."

print("Baseline:", interpret_history(baseline_history))
print("Dropout Adam:", interpret_history(dropout_history))
print("Dropout SGD:", interpret_history(sgd_history))

Baseline: Convergence improved because training and validation loss decreased.
Dropout Adam: Convergence improved because training and validation loss decreased.
Dropout SGD: Convergence improved because training and validation loss decreased.

Use this summary to decide whether dropout helped reduce overfitting or whether optimizer changes improved convergence speed.



<a href="https://dataplatform.cloud.ibm.com/registration/stepone?utm_source=skills_network&utm_content=in_lab_content_link&utm_id=Lab-IBMDeveloperSkillsNetwork-DL0110EN-SkillsNetwork&context=cpdaas&apps=data_science_experience%2Cwatson_machine_learning"><img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DL0110EN-SkillsNetwork/Template/module%201/images/Watson_Studio.png"></a>


<!--Empty Space for separating topics-->


<h2>About the Author:</h2> 

**Harish Pant** is an accomplished IT Consultant specializing in bridging the gap between complex AI technologies and practical business applications. Holding a Bachelor’s degree in Computer Science with a strong background in Machine Learning, Harish serves clients across diverse industries—including fintech, retail, and manufacturing—helping them navigate AI adoption. Harish is known for translating technical AI concepts into actionable business strategies and delivering engaging, high-impact presentations.


<!--
## Change Log

|  Date (YYYY-MM-DD) |  Version | Changed By  |  Change Description |
|---|---|---|---|
| 2026-05-11  | 1.0  | IBM Skills Network  |  Created lab for improving an RNN sentiment classifier with regularization and tuning |
-->


<hr>





## <h3 align="center"> &#169; IBM Corporation. All rights reserved. <h3/>
